In [2]:
import os

MCP_URL = os.environ.get("MCP_URL")
MODEL = os.environ.get("OPENAI_MODEL")

print("MCP URL: ", MCP_URL)
print("MODEL", MODEL)
print("API KEY: ", "set" if os.environ.get("OPENAI_API_KEY") else "MISSING")

MCP URL:  None
MODEL gpt-5.6-terra
API KEY:  set


In [10]:
from agents.extensions.models.litellm_model import LitellmModel
from agents.mcp import MCPServerStreamableHttp
from agents import Runner, Agent
from agents.model_settings import ModelSettings
from openai.types.shared import Reasoning

In [4]:
model = LitellmModel(MODEL)

In [31]:
agent = Agent(
    name=" assistant",
    instructions = """
        You are a data analyst assistant with access to a postgres database via MCP to complete Text-To-Sql tasks.
        Please refer to the MCP tools that you have available to return the most relevant answer for the user
    """,
    model = model,
    model_settings=ModelSettings(reasoning=Reasoning(effort="none")),
    hooks=ThoughtTracker(),
)

In [12]:
async with MCPServerStreamableHttp(
    name="Postgres MCP Server",
    params={
        "url": "http://mcp:8000/mcp",
        "timeout": 120
    },
    client_session_timeout_seconds = 120,
    cache_tools_list=True
) as server:
    agent = agent.clone(mcp_servers=[server])
    result = await Runner.run(agent, "Ce tool-uri ai? Cate tool-uri sunt native si care sunt custom?")

In [13]:
print(result)

RunResult:
- Last agent: Agent(name="Database assistant", ...)
- Final output (str):
    Am acces la următoarele tool-uri:
    
    ### Tool-uri native (7)
    1. `get_schema_names` — listează schemele din baza de date  
    2. `get_tables` — listează tabelele dintr-o schemă  
    3. `get_columns` — afișează coloanele unui tabel  
    4. `get_indexes` — afișează indexurile unui tabel  
    5. `get_foreign_keys` — afișează cheile străine  
    6. `run_dql_query` — rulează interogări de citire SQL, de tip `SELECT`  
    7. `run_ddl_query` — rulează comenzi de definire structură, precum `CREATE` / `ALTER` / `DROP`  
    8. `run_dml_query` — rulează modificări de date, precum `INSERT` / `UPDATE` / `DELETE`  
    9. `run_dcl_query` — rulează comenzi de permisiuni, precum `GRANT` / `REVOKE`  
    
    ### Tool-uri custom (1)
    10. `top_customers` — returnează clienții de top după venitul total, cu filtre pentru limită și statusul comenzilor.
    
    În total sunt **10 tool-uri de bază**: 

In [32]:
async with MCPServerStreamableHttp(
    name="Postgres MCP Server",
    params={
        "url": "http://mcp:8000/mcp",
        "timeout": 120
    },
    client_session_timeout_seconds = 120,
    cache_tools_list=True
) as server:
    agent = agent.clone(mcp_servers=[server])
    result = await Runner.run(agent, "Care este valoarea medie a comenzii pe fiecare loyalty tier? Livrate si expediate.")

Agent Database assistant starting...
LLM response received!
---------------------------------------------
Agent calling tool: FunctionTool(name='get_schema_names', description='Get all schema names.', params_json_schema={'properties': {}, 'title': 'get_schema_namesArguments', 'type': 'object'}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x7fcf9fc71bd0>, strict_json_schema=False, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False, custom_data_extractor=None, allowed_callers=None, output_json_schema=None) | MCP: Postgres MCP Server
Tool: get_schema_names completed...
LLM response received!
---------------------------------------------
Agent calling tool: FunctionTool(name='get_tables', description='Get all tables in a schema.\n\n    Args:\n        schema_name: The name of the schema to get tables from, defau

In [33]:
print(result)

RunResult:
- Last agent: Agent(name="Database assistant", ...)
- Final output (str):
    Valoarea medie a comenzii pentru comenzile **livrate** și **expediate**:
    
    | Loyalty tier | Valoare medie comandă | Nr. comenzi |
    |---|---:|---:|
    | bronze | 164,24 | 137 |
    | silver | 174,73 | 105 |
    | gold | 179,04 | 76 |
    | platinum | 182,59 | 52 |
    
    ```sql
    SELECT
      c.loyalty_tier,
      ROUND(AVG(o.total_amount), 2) AS valoare_medie_comanda,
      COUNT(*) AS numar_comenzi
    FROM public.orders AS o
    JOIN public.customers AS c
      ON c.customer_id = o.customer_id
    WHERE o.status IN ('delivered', 'shipped')
    GROUP BY c.loyalty_tier
    ORDER BY c.loyalty_tier;
    ```
- 11 new item(s)
- 5 raw response(s)
- 0 input guardrail result(s)
- 0 output guardrail result(s)
(See `RunResult` for more details)


In [16]:
from agents import AgentHooks, RunContextWrapper, FunctionToolResult
from agents.tool import get_function_tool_origin
from typing import Union, Any, List, Optional

In [26]:
class ThoughtTracker(AgentHooks):
    def __init__(self):
        self.total_input_tokens = 0
        self.total_output_tokens = 0

    async def on_start(self, context: RunContextWrapper, agent) -> None:
        print(f"Agent {agent.name} starting...")

    async def on_tool_start(self, context: RunContextWrapper, agent, tool: Union[Any]) -> None: 
        print("---------------------------------------------")
        tool_name = getattr(tool, 'name', str(tool))
        mcp_server = get_function_tool_origin(tool).mcp_server_name
        print(f"Agent calling tool: {tool} | MCP: {mcp_server}")
        
    async def on_tool_end(self, context: RunContextWrapper, agent, tool: Union[Any], result: str) -> None:
        tool_name = getattr(tool, 'name', str(tool))
        print(f"Tool: {tool_name} completed...")

    async def on_llm_start(self, context: RunContextWrapper, agent, system_prompt: Optional[str], input_items: List[Any]) -> None:
        print("---------------------------------------------")
        print(f"LLM processing {len(input_items)} items...")

    async def on_llm_end(self, context: RunContextWrapper, agent, response):
        print("LLM response received!")
        if hasattr(response, 'usage') and response.usage:
            self.total_input_tokens += response.usage.input_tokens
            self.total_output_tokens += response.usage.output_tokens

    async def on_end(self, context: RunContextWrapper, agent, output: Any) -> None:
        print(f"Agent {agent.name} finished!")
        total_tokens = self.total_input_tokens + self.total_output_tokens
        input_cost = (self.total_input_tokens / 1_000_000) *  2
        output_cost = (self.total_output_tokens / 1_000_000) * 12
        total_cost = input_cost + output_cost

        print("---------------------------------------------")
        print("Cost summary")
        print(f"Input tokens: {self.total_input_tokens} ${input_cost:.4f}")
        print(f"Output tokens: {self.total_output_tokens} ${output_cost:.4f}")
        print(f"Total cost: ${total_cost:.4f}")

In [34]:
async with MCPServerStreamableHttp(
    name="Postgres MCP Server",
    params={
        "url": "http://mcp:8000/mcp",
        "timeout": 120
    },
    client_session_timeout_seconds = 120,
    cache_tools_list=True
) as server:
    agent = agent.clone(mcp_servers=[server])
    result = await Runner.run(agent, "Valoarea totala a produselor vandute, grupate per categorie")

Agent Database assistant starting...
LLM response received!
---------------------------------------------
Agent calling tool: FunctionTool(name='get_tables', description='Get all tables in a schema.\n\n    Args:\n        schema_name: The name of the schema to get tables from, defaults to "public".\n    ', params_json_schema={'properties': {'schema_name': {'anyOf': [{'type': 'string'}, {'type': 'null'}], 'default': 'public', 'title': 'Schema Name'}}, 'title': 'get_tablesArguments', 'type': 'object'}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x7fcf9f839cd0>, strict_json_schema=False, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False, custom_data_extractor=None, allowed_callers=None, output_json_schema=None) | MCP: Postgres MCP Server
Tool: get_tables completed...
LLM response received!
-------------------

In [36]:
print(result)

RunResult:
- Last agent: Agent(name="Database assistant", ...)
- Final output (str):
    | Categorie | Valoare totală vândută |
    |---|---:|
    | Electronics | 24.651,00 |
    | Home | 19.543,80 |
    | Garden | 10.152,00 |
    | Sports | 9.926,87 |
    | Fashion | 8.566,74 |
    | Toys | 7.026,67 |
    | Books | 4.835,37 |
- 9 new item(s)
- 4 raw response(s)
- 0 input guardrail result(s)
- 0 output guardrail result(s)
(See `RunResult` for more details)
